<a href="https://colab.research.google.com/github/Akanksha24-nema/Multi_agent_system_using_GeminiAPI/blob/main/Multi_agent_system_using_GeminiAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-genai

In [ ]:
import os
from google.colab import userdata
import google.genai as genai

def configure():
  secret_name= "Gemini_API"

  API_KEY = userdata.get(secret_name)
  if not API_KEY:
    raise ValueError(f"Secret '{secret_name}' not found")

  client = genai.Client(api_key=API_KEY)
  return client

In [ ]:
def planner_agent(client, topic: str)-> list[str]:
  print ("Planner_agent: Creating a research plan")
  prompt = f"""
  You are an expert research planner. Your task is to break down the following topic into 3-5 specific, answerable questions. Return these questions as a Python list of strings.

  TOPIC : {topic}

  Example output: ["question 1","question 2", "question 3"]
  """
  try:
     response = client.models.generate_content(
     model = "gemini-3.1-flash-lite",
     contents = prompt
     )

     plan_str = response.text.strip().replace('[','').replace( ']','').replace('"','')
     plan = [q.strip() for q in plan_str.split(',') if q.strip()]

     print('plan created')
     for i, q in enumerate(plan,1):
      print(f"{i}.{q}")
     return plan

  except Exception as e:
    print(f"error in planner agent as {e}")
    return []

In [ ]:
from google.genai import types

def search_agent(client, question: str)-> str:
  print(f"Search agent is Researching question:'{question}'....")

  try:
    search_tool= types.Tool(
        google_search= types.GoogleSearch()
        )
    prompt = f"Provide a detailed answer to the following question:{question}"
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config = types.GenerateContentConfig(
            tools = [search_tool]
        )
    )
    print("Answer found")
    return response.text

  except Exception as e:
    print(f"Error in finding answer to the question as :{e}")
    return ''

In [ ]:
def synthesizer_agent(client, topic:str, research_result: list)-> str:
  print(f"Researching and generating the final report on the given topic titled as : {topic} ")
  research_notes = ""
  for question, data in research_result:
    research_notes += f"""Question{question}\n###Research Data :\n{data}\n\n---\n\n"""

  prompt=f"""You are an expert research analyst. Your task is to synthesize the provided research notes into a comprehensive, well-structured report on the topic: "{topic}".
  The report should have an introduction, a body that covers the key findings from the notes,and a conclusion. Use the information from the research notes ONLY.

  ##Reasearch_notes##
  {research_notes}
  """

  try:
    response = client.models.generate_content(
        model = "gemini-3.1-flash-lite",
        contents=prompt,
        )
    return response.text

  except Exception as e:
    print(f" Error in synthesizer agent:{e}")
    return "Error: Could not genrate the final report"


In [ ]:
def main():
  try:
    client= configure()
  except ValueError as e:
    print(e)
    return

  print(f"\n Hello I am your research assistant...")
  topic=input("Tell me what I shall research upon today..?")

  if not topic.strip():
    print("A topic is required to begin research.. Exiting!!")
    return

  print(f"\n Starting Research on : {topic}")

  research_plan= planner_agent(client, topic)
  if not research_plan:
    print("Can't create a research plan, Exiting!!!")
    return

  research_result= []
  for question in research_plan:
    research_data= search_agent(client, question) # Corrected typo: seacrh_agent to search_agent
    if research_data:
      research_result.append((question, research_data)) # Corrected typo: reseacrh_result to research_result

    if not research_data:
      print(f"No relevant data found for information on the give:'{topic}'")
      return

  final_report = synthesizer_agent(client, topic, research_result)

  print("\n****FINAL RESEARCH****")
  print(f"topic:'{topic}'")
  print(final_report)
  print("\n END OF THE REPORT")


if __name__ == "__main__":
   main()


 Hello I am your research assistant...
Tell me what I shall research upon today..?Ambey Electicals, Ahinsa Chowk Jabalpur

 Starting Research on : Ambey Electicals, Ahinsa Chowk Jabalpur
Planner_agent: Creating a research plan
plan created
1.What is the official contact number and physical address of Ambey Electricals located at Ahinsa Chowk
2.Jabalpur?
3.What specific types of electrical products or services does Ambey Electricals offer to its customers?
4.What are the typical operating hours for Ambey Electricals at Ahinsa Chowk?
5.Does Ambey Electricals have an online presence or customer reviews available on platforms like Google Maps or Justdial?
Search agent is Researching question:'What is the official contact number and physical address of Ambey Electricals located at Ahinsa Chowk'....
Error in finding answer to the question as :429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more info